# Standard Notebook for copying

## Origin of Files

In [1]:
#Short history of commands and origin of files:
# The commands are for the PI dataset, the other datasets have been created exactly analogous 
# just the names of the files have been exchanged, they can be found in the folders
# /data/modeldata/ICON/icon-paleo/LIG.kepler/remapped_r2b4  &  ./data/modeldata/ICON/icon-paleo/LIG.kepler.ghg/remapped_r2b4

#
#Merging:
#        cdo mergetime ../PI.kepler.mk_atm_2d_ml_*.nc PI.kepler_atm2d_merged.nc
#
#Regridding:
#        getting gridfiles/icon_grid_0012_R02B04_G.nc
#                ==> downloaded original from ICON paleo website, is the gridfile for R02B04 grid
#        creating gridfiles/target_grid_1_5.txt
#                ==> got from Kiras 0.75 grid, changed numbers pretty self explanatory!
#
#        Creating weightfile in gridfiles:
#        cdo gendis,gridfiles/target_grid_1_5.txt -setgrid,gridfiles/icon_grid_0012_R02B04_G.nc  PI.kepler_atm2d_merged.nc gridfiles/weightfile_R02B04_to_1_5_deg
#
#        Actual Remapping:
#        cdo -remap,gridfiles/target_grid_1_5.txt,gridfiles/weightfile_R02B04_to_1_5_deg PI.kepler_atm2d_merged.nc PI.kepler_atm2d_merged_remapped.nc
#
#Selecting the relevant years:
#        cdo selyear,1770/1800 PI.kepler_atm2d_merged_remapped.nc PI.kepler_atm2d_merged_remapped_1770_1800.nc
#
#
#Extracting DIFFERENT data:
#
#        Creating Means:
#        cdo ymonmean PI.kepler_atm2d_merged_remapped_1770_1800.nc ymonmean_PI.kepler_atm2d_merged_remapped_1770_1800.nc
#
#        cdo timmean PI.kepler_atm2d_merged_remapped_1770_1800.nc timmean_PI.kepler_atm2d_merged_remapped_1770_1800.nc
#
#        Isolating Temperature and Sea Ice:
#        for whole data
#        cdo selname,t_2m,fr_seaice PI.kepler_atm2d_merged_remapped.nc PI.kepler_t2m_sif_merged_remapped.nc
#        for 1770-1800:
#        cdo selname,t_2m,fr_seaice PI.kepler_atm2d_merged_remapped_1770_1800.nc PI.kepler_t2m_sif_merged_remapped_1770_1800.nc
#


## Basic Import Statements

In [2]:
# %%
import xarray as xr
import numpy as np
#from scipy.stats import pearsonr
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
#from tqdm import tqdm
import os
import socket
import psutil
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## Loading rcParams

In [3]:
import importlib
import plot_settings
importlib.reload(plot_settings)
plot_settings.load_rcParams()

## Loading Data

In [5]:
def determine_system() -> str:
    many_cores  = os.cpu_count() > 16
    lots_of_ram = psutil.virtual_memory().total > 64 * 1024**3  # more than 64 GB
    if many_cores and lots_of_ram:
        return 'Cluster'
    else:
        return 'Jakob_Laptop'

if determine_system() == 'Jakob_Laptop':
    location_of_data = '../'
else:
    location_of_data = '/data/modeldata/ICON/icon-paleo/'

PI      = xr.open_dataset(location_of_data + 'PI.kepler/remapped_r2b4/PI.kepler_t2m_sif_merged_remapped.nc')
LIG     = xr.open_dataset(location_of_data + 'LIG.kepler/remapped_r2b4/LIG.kepler_t2m_sif_merged_remapped.nc')
LIG_ghg = xr.open_dataset(location_of_data + 'LIG.kepler.ghg/remapped_r2b4/LIG.kepler.ghg_t2m_sif_merged_remapped.nc')


In [31]:
def get_weighted_mean(PI: xr.DataArray) -> xr.DataArray:
    PI = PI.squeeze('height_2')
    weights = get_lat_weights(PI)
    PI_lat_weighted = PI.weighted(weights)
    
    return PI_lat_weighted.mean(dim=['lat', 'lon'])

In [32]:
PI_annual = get_annual_mean_from_raw_dataset(PI)
#LIG_annual = get_annual_mean_from_raw_dataset(LIG)
#LIG_ghg_annual = get_annual_mean_from_raw_dataset(LIG_ghg)

